<a href="https://colab.research.google.com/github/Priti02707/Fake-News-Detection/blob/main/Fake_News_Detection_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.layers import Dense, Dropout

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [25]:
fake = pd.read_csv(
    "/content/Fake.csv",
    on_bad_lines="skip",
    engine="python"
)

real = pd.read_csv(
    "/content/True.csv",
    on_bad_lines="skip",
    engine="python"
)

print("Fake dataset:", fake.shape)
print("Real dataset:", real.shape)

Fake dataset: (23481, 4)
Real dataset: (21417, 4)


In [26]:
fake["label"] = 0
real["label"] = 1

print("Fake label:", fake["label"].unique())
print("Real label:", real["label"].unique())

Fake label: [0]
Real label: [1]


In [27]:
df = pd.concat(
    [fake, real],
    axis=0
)

df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Final Dataset Shape:", df.shape)

print("\nLabel Distribution:")
print(df["label"].value_counts())

Final Dataset Shape: (44898, 5)

Label Distribution:
label
0    23481
1    21417
Name: count, dtype: int64


In [28]:
df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)

print("Missing values after cleaning:")
print(df[["title", "text"]].isnull().sum())

Missing values after cleaning:
title    0
text     0
dtype: int64


In [29]:
df["content"] = (
    df["title"].str.strip()
    + " "
    + df["text"].str.strip()
)

print(df[["title", "text", "content", "label"]].head())

                                               title  \
0  Ben Stein Calls Out 9th Circuit Court: Committ...   
1  Trump drops Steve Bannon from National Securit...   
2  Puerto Rico expects U.S. to lift Jones Act shi...   
3   OOPS: Trump Just Accidentally Confirmed He Le...   
4  Donald Trump heads for Scotland to reopen a go...   

                                                text  \
0  21st Century Wire says Ben Stein, reputable pr...   
1  WASHINGTON (Reuters) - U.S. President Donald T...   
2  (Reuters) - Puerto Rico Governor Ricardo Rosse...   
3  On Monday, Donald Trump once again embarrassed...   
4  GLASGOW, Scotland (Reuters) - Most U.S. presid...   

                                             content  label  
0  Ben Stein Calls Out 9th Circuit Court: Committ...      0  
1  Trump drops Steve Bannon from National Securit...      1  
2  Puerto Rico expects U.S. to lift Jones Act shi...      1  
3  OOPS: Trump Just Accidentally Confirmed He Lea...      0  
4  Donald Trump 

In [30]:
def clean_text(text):
    text = text.lower()

    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["content"] = df["content"].apply(clean_text)
print(df["content"].head())

0    ben stein calls out th circuit court committed...
1    trump drops steve bannon from national securit...
2    puerto rico expects u s to lift jones act ship...
3    oops trump just accidentally confirmed he leak...
4    donald trump heads for scotland to reopen a go...
Name: content, dtype: object


In [31]:
print("Final Shape:", df.shape)
print("\nFake:", (df["label"] == 0).sum())
print("Real:", (df["label"] == 1).sum())
print("\nMissing content:", df["content"].isnull().sum())

Final Shape: (44898, 6)

Fake: 23481
Real: 21417

Missing content: 0


In [32]:
X = df["content"]
y = df["label"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train_text))
print("Testing samples:", len(X_test_text))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

Training samples: 35918
Testing samples: 8980

Training label distribution:
label
0    18785
1    17133
Name: count, dtype: int64

Testing label distribution:
label
0    4696
1    4284
Name: count, dtype: int64


In [33]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_df=0.8,
    min_df=5,
    ngram_range=(1, 2),
    max_features=50000
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("Training TF-IDF Shape:", X_train_tfidf.shape)
print("Testing TF-IDF Shape:", X_test_tfidf.shape)

Training TF-IDF Shape: (35918, 50000)
Testing TF-IDF Shape: (8980, 50000)


In [34]:
lr_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced"
)

lr_model.fit(X_train_tfidf, y_train)

lr_pred = lr_model.predict(X_test_tfidf)

lr_accuracy = accuracy_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)

print("========== Logistic Regression ==========")
print("Accuracy:", lr_accuracy)
print("F1 Score:", lr_f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    lr_pred,
    target_names=["Fake", "Real"]
))

========== Logistic Regression ==========
Accuracy: 0.9874164810690423
F1 Score: 0.9868742014171216

Classification Report:
              precision    recall  f1-score   support

        Fake       0.99      0.98      0.99      4696
        Real       0.98      0.99      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [35]:
svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=5000
)

svm_model.fit(X_train_tfidf, y_train)

svm_pred = svm_model.predict(X_test_tfidf)

svm_accuracy = accuracy_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred)

print("========== SVM ==========")
print("Accuracy:", svm_accuracy)
print("F1 Score:", svm_f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    svm_pred,
    target_names=["Fake", "Real"]
))

========== SVM ==========
Accuracy: 0.9956570155902005
F1 Score: 0.9954540156195361

Classification Report:
              precision    recall  f1-score   support

        Fake       1.00      0.99      1.00      4696
        Real       0.99      1.00      1.00      4284

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980



In [36]:
MAX_WORDS = 20000
MAX_LENGTH = 300

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

# Only training data par tokenizer fit hoga
tokenizer.fit_on_texts(X_train_text)

# Text → sequences
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

# Same length ke sequences
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("LSTM Training Shape:", X_train_pad.shape)
print("LSTM Testing Shape:", X_test_pad.shape)

LSTM Training Shape: (35918, 300)
LSTM Testing Shape: (8980, 300)


In [37]:
lstm_model = Sequential([

    Embedding(
        input_dim=MAX_WORDS,
        output_dim=128,
        input_length=MAX_LENGTH
    ),

    LSTM(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_model.build(input_shape=(None, MAX_LENGTH))

lstm_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 300, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,611,521 (9.96 MB)

 Trainable params: 2,611,521 (9.96 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history_lstm = lstm_model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
271/898 ━━━━━━━━━━━━━━━━━━━━ 2:20 224ms/step - accuracy: 0.6555 - loss: 0.6202

In [ ]:
lstm_prob = lstm_model.predict(X_test_pad, verbose=0)

lstm_pred = (lstm_prob >= 0.5).astype(int).flatten()

lstm_accuracy = accuracy_score(y_test, lstm_pred)
lstm_f1 = f1_score(y_test, lstm_pred)

print("========== LSTM ==========")
print("Accuracy:", lstm_accuracy)
print("F1 Score:", lstm_f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    lstm_pred,
    target_names=["Fake", "Real"]
))

In [ ]:
cnn_model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=128,
        input_length=MAX_LENGTH
    ),

    Conv1D(
        filters=128,
        kernel_size=5,
        activation="relu"
    ),

    GlobalMaxPooling1D(),

    Dense(64, activation="relu"),

    Dropout(0.5),

    Dense(1, activation="sigmoid")
])

cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn_model.build(input_shape=(None, MAX_LENGTH))

cnn_model.summary()

In [ ]:
history_cnn = cnn_model.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=32,
    verbose=1
)

In [ ]:
cnn_prob = cnn_model.predict(X_test_pad, verbose=0)

cnn_pred = (cnn_prob >= 0.5).astype(int).flatten()

cnn_accuracy = accuracy_score(y_test, cnn_pred)
cnn_f1 = f1_score(y_test, cnn_pred)

print("========== 1D-CNN ==========")
print("Accuracy:", cnn_accuracy)
print("F1 Score:", cnn_f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    cnn_pred,
    target_names=["Fake", "Real"]
))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "SVM",
        "Logistic Regression",
        "LSTM",
        "1D-CNN"
    ],
    "Accuracy": [
        svm_accuracy,
        lr_accuracy,
        lstm_accuracy,
        cnn_accuracy
    ],
    "F1 Score": [
        svm_f1,
        lr_f1,
        lstm_f1,
        cnn_f1
    ]
})

results["Accuracy (%)"] = results["Accuracy"] * 100
results["F1 Score (%)"] = results["F1 Score"] * 100

results = results.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

print(results)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.arange(len(results["Model"]))
width = 0.35

plt.figure(figsize=(10, 6))

bars1 = plt.bar(
    x - width/2,
    results["Accuracy (%)"],
    width,
    label="Accuracy"
)

bars2 = plt.bar(
    x + width/2,
    results["F1 Score (%)"],
    width,
    label="F1 Score"
)

plt.xlabel("Models")
plt.ylabel("Score (%)")
plt.title("Accuracy and F1 Score Comparison")
plt.xticks(x, results["Model"], rotation=20)
plt.ylim(0, 105)
plt.legend()

# Show exact values on bars
for bar in bars1:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.5,
        f"{height:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )

for bar in bars2:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.5,
        f"{height:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Create confusion matrices
cm_svm = confusion_matrix(y_test, svm_pred)
cm_lr = confusion_matrix(y_test, lr_pred)
cm_lstm = confusion_matrix(y_test, lstm_pred)
cm_cnn = confusion_matrix(y_test, cnn_pred)

# Plot all 4 confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# SVM
sns.heatmap(
    cm_svm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fake", "Real"],
    yticklabels=["Fake", "Real"],
    ax=axes[0, 0]
)
axes[0, 0].set_title("SVM")
axes[0, 0].set_xlabel("Predicted")
axes[0, 0].set_ylabel("Actual")

# Logistic Regression
sns.heatmap(
    cm_lr,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fake", "Real"],
    yticklabels=["Fake", "Real"],
    ax=axes[0, 1]
)
axes[0, 1].set_title("Logistic Regression")
axes[0, 1].set_xlabel("Predicted")
axes[0, 1].set_ylabel("Actual")

# LSTM
sns.heatmap(
    cm_lstm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fake", "Real"],
    yticklabels=["Fake", "Real"],
    ax=axes[1, 0]
)
axes[1, 0].set_title("LSTM")
axes[1, 0].set_xlabel("Predicted")
axes[1, 0].set_ylabel("Actual")

# 1D-CNN
sns.heatmap(
    cm_cnn,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fake", "Real"],
    yticklabels=["Fake", "Real"],
    ax=axes[1, 1]
)
axes[1, 1].set_title("1D-CNN")
axes[1, 1].set_xlabel("Predicted")
axes[1, 1].set_ylabel("Actual")

plt.tight_layout()
plt.show()